# Offre de médecins généralistes et demande de soins

Ce notebook prépare les deux composantes nécessaires au calcul de l'indicateur d'accessibilité :

- **offre** : nombre de médecins généralistes par commune ;
- **demande** : population communale standardisée selon la structure par âge.

## Objectif

Produire une table avec une ligne par commune :

```text
CODGEO
LIBGEO
population
nb_medecins_generalistes
demande_standardisee
```

La table obtenue sera ensuite utilisée avec la table de distances.



In [19]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


In [20]:
BASE_PUBLIC = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"

URL_INSEE = BASE_PUBLIC + "insee_communes_final.csv"
URL_PERSONNE = BASE_PUBLIC + "personne_activite_clean.csv"
URL_SAVOIRFAIRE = BASE_PUBLIC + "savoirfaire_clean.csv"

DOSSIER_LOCAL = Path("data/processed")
DOSSIER_LOCAL.mkdir(parents=True, exist_ok=True)

OUTPUT_LOCAL = DOSSIER_LOCAL / "offre_demande_mg_commune.parquet"
OUTPUT_S3 = "s3://mangane/Projet_PDS_2026/processed/offre_demande_mg_commune.parquet"
UPLOAD_S3 = True


In [21]:
insee = pd.read_csv(URL_INSEE, dtype={"CODGEO": str}, low_memory=False)
personne = pd.read_csv(URL_PERSONNE, dtype={"Identifiant PP": str, "Code commune (coord. structure)": str}, low_memory=False)
savoirfaire = pd.read_csv(URL_SAVOIRFAIRE, dtype={"Identifiant PP": str}, low_memory=False)

print("INSEE :", insee.shape)
print("Personne-activité :", personne.shape)
print("Savoir-faire :", savoirfaire.shape)


INSEE : (34858, 35)
Personne-activité : (2313070, 12)
Savoir-faire : (434937, 6)


In [22]:
CHANGEMENTS_CODES = {
    "12076": "12218",
    "14011": "14581",
    "49069": "49126",
    "69159": "69114",
}

def harmoniser_codgeo(code):
    if pd.isna(code):
        return pd.NA
    code = str(code).strip()
    if re.fullmatch(r"751(?:0[1-9]|1[0-9]|20)", code):
        return "75056"
    if re.fullmatch(r"132(?:0[1-9]|1[0-6])", code):
        return "13055"
    if re.fullmatch(r"6938[1-9]", code):
        return "69123"
    return CHANGEMENTS_CODES.get(code, code)


# A. Offre de médecins généralistes

In [23]:
def normaliser_texte(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"\s+", " ", x)
    return x

savoirfaire["profession_norm"] = savoirfaire["Libellé profession"].map(normaliser_texte)
savoirfaire["savoirfaire_norm"] = savoirfaire["Libellé savoir-faire"].map(normaliser_texte)

print(
    savoirfaire.loc[
        savoirfaire["savoirfaire_norm"].str.contains("general", na=False),
        "Libellé savoir-faire"
    ].dropna().value_counts().head(30)
)


Libellé savoir-faire
Spécialiste en Médecine Générale              90353
Qualifié en Médecine Générale                 53795
Médecine Générale                              8875
Chirurgie générale                             4102
Biologie médicale option biologie générale       81
Name: count, dtype: int64


In [24]:
masque_medecin = savoirfaire["profession_norm"].str.contains(r"\bmedecin\b", regex=True, na=False)
masque_generaliste = savoirfaire["savoirfaire_norm"].str.contains(r"medecine generale|medecin generaliste", regex=True, na=False)

ids_mg = set(
    savoirfaire.loc[masque_medecin & masque_generaliste, "Identifiant PP"]
    .dropna().astype(str)
)

print("Médecins généralistes distincts identifiés :", len(ids_mg))


Médecins généralistes distincts identifiés : 152765


In [25]:
mg_activite = personne[
    personne["Identifiant PP"].isin(ids_mg)
].copy()

mg_activite["CODGEO"] = (
    mg_activite["Code commune (coord. structure)"]
    .map(harmoniser_codgeo)
)

mg_activite = mg_activite.dropna(
    subset=["CODGEO", "Identifiant PP"]
)

# Un médecin compte qu'une fois par commune
mg_commune = (
    mg_activite[
        ["Identifiant PP", "CODGEO"]
    ]
    .drop_duplicates()
)

# Nombre de communes d'exercice par médecin
mg_commune["nb_communes_exercice"] = (
    mg_commune
    .groupby("Identifiant PP")["CODGEO"]
    .transform("nunique")
)

# Répartition du médecin entre ses communes
mg_commune["poids_medecin"] = (
    1 / mg_commune["nb_communes_exercice"]
)

# Offre équivalente par commune
offre_mg = (
    mg_commune
    .groupby("CODGEO", as_index=False)
    ["poids_medecin"]
    .sum()
    .rename(
        columns={
            "poids_medecin":
            "nb_medecins_generalistes"
        }
    )
)

print(
    "Médecins uniques :",
    mg_commune["Identifiant PP"].nunique()
)

print(
    "Somme de l'offre :",
    offre_mg["nb_medecins_generalistes"].sum()
)

Médecins uniques : 94527
Somme de l'offre : 94527.0


In [26]:
print("MG identifiés :", len(ids_mg))

print(
    "MG uniques avec activité :",
    mg_activite["Identifiant PP"].nunique()
)

print(
    "Couples médecin-commune :",
    mg_activite[
        ["Identifiant PP", "CODGEO"]
    ].drop_duplicates().shape[0]
)

print(
    "Somme après agrégation communale :",
    offre_mg["nb_medecins_generalistes"].sum()
)

MG identifiés : 152765
MG uniques avec activité : 94527
Couples médecin-commune : 112532
Somme après agrégation communale : 94527.0


# B. Demande standardisée

In [27]:
classes_age = {
    "pop_0_2": ["SEXE1_AGEPYR1000", "SEXE2_AGEPYR1000"],
    "pop_3_5": ["SEXE1_AGEPYR1003", "SEXE2_AGEPYR1003"],
    "pop_6_10": ["SEXE1_AGEPYR1006", "SEXE2_AGEPYR1006"],
    "pop_11_17": ["SEXE1_AGEPYR1011", "SEXE2_AGEPYR1011"],
    "pop_18_24": ["SEXE1_AGEPYR1018", "SEXE2_AGEPYR1018"],
    "pop_25_39": ["SEXE1_AGEPYR1025", "SEXE2_AGEPYR1025"],
    "pop_40_54": ["SEXE1_AGEPYR1040", "SEXE2_AGEPYR1040"],
    "pop_55_64": ["SEXE1_AGEPYR1055", "SEXE2_AGEPYR1055"],
    "pop_65_79": ["SEXE1_AGEPYR1065", "SEXE2_AGEPYR1065"],
    "pop_80_plus": ["SEXE1_AGEPYR1080", "SEXE2_AGEPYR1080"],
}

colonnes_absentes = [col for cols in classes_age.values() for col in cols if col not in insee.columns]
print("Colonnes d'âge absentes :", colonnes_absentes)
assert not colonnes_absentes

demande = insee[["CODGEO", "LIBGEO"]].copy()
for nouvelle_colonne, colonnes in classes_age.items():
    demande[nouvelle_colonne] = insee[colonnes].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)

colonnes_age = list(classes_age.keys())
demande["population"] = demande[colonnes_age].sum(axis=1, min_count=1)
demande.head()


Colonnes d'âge absentes : []


,CODGEO,LIBGEO,pop_0_2,pop_3_5,pop_6_10,pop_11_17,pop_18_24,pop_25_39,pop_40_54,pop_55_64,pop_65_79,pop_80_plus,population
0,01001,L'Abergement-Clémenciat,33.000000,28.000000,56.000000,81.000000,30.000000,151.000000,177.000000,131.000000,128.000000,44.000000,859.0
1,01002,L'Abergement-de-Varey,6.000000,12.000000,19.000000,34.000000,11.000000,41.000000,74.000000,26.000000,32.000000,18.000000,273.0
2,01004,Ambérieu-en-Bugey,615.743522,610.563188,992.962424,1384.174340,1470.391014,3099.339484,2753.287276,1818.260858,2040.179086,769.098809,15554.0
3,01005,Ambérieux-en-Dombes,65.374296,70.908450,127.068163,161.666068,107.724534,389.045273,400.780251,268.831175,239.134291,86.467498,1917.0
4,01006,Ambléon,3.000000,3.000000,4.000000,2.000000,6.000000,22.000000,18.000000,24.000000,26.000000,6.000000,114.0



Les classes INSEE sont plus larges que les classes quinquennales de la DREES. Les poids ci-dessous sont donc des **poids agrégés approximatifs**, construits à partir des poids historiques DREES, puis la demande est renormalisée au niveau national.


In [28]:
poids_age = {
    "pop_0_2": 1.27,
    "pop_3_5": (2 * 1.27 + 1 * 0.65) / 3,
    "pop_6_10": (4 * 0.65 + 1 * 0.55) / 5,
    "pop_11_17": (4 * 0.55 + 3 * 0.51) / 7,
    "pop_18_24": (2 * 0.51 + 5 * 0.54) / 7,
    "pop_25_39": (5 * 0.65 + 5 * 0.67 + 5 * 0.72) / 15,
    "pop_40_54": (5 * 0.75 + 5 * 0.83 + 5 * 1.42) / 15,
    "pop_55_64": (5 * 1.07 + 5 * 1.31) / 10,
    "pop_65_79": (5 * 1.37 + 5 * 1.54 + 5 * 1.82) / 15,
    "pop_80_plus": (2.12 + 2.84 + 2.61) / 3,
}

pd.Series(poids_age, name="poids").round(3)


pop_0_2        1.270
pop_3_5        1.063
pop_6_10       0.630
pop_11_17      0.533
pop_18_24      0.531
pop_25_39      0.680
pop_40_54      1.000
pop_55_64      1.190
pop_65_79      1.577
pop_80_plus    2.523
Name: poids, dtype: float64

In [29]:
demande["demande_ponderee_brute"] = 0.0
for col, poids in poids_age.items():
    demande["demande_ponderee_brute"] += demande[col].fillna(0) * poids

population_nationale = demande["population"].sum()
demande_ponderee_nationale = demande["demande_ponderee_brute"].sum()
facteur_normalisation = population_nationale / demande_ponderee_nationale

demande["demande_standardisee"] = demande["demande_ponderee_brute"] * facteur_normalisation

print("Population totale :", population_nationale)
print("Demande standardisée totale :", demande["demande_standardisee"].sum())
print("Facteur de normalisation :", facteur_normalisation)


Population totale : 67760573.0
Demande standardisée totale : 67760573.0
Facteur de normalisation : 0.9488595246998868


# C. Fusion offre + demande

In [30]:
offre_demande = demande.merge(offre_mg, on="CODGEO", how="left", validate="one_to_one")
offre_demande["nb_medecins_generalistes"] = offre_demande["nb_medecins_generalistes"].fillna(0)

assert not offre_demande["CODGEO"].duplicated().any()

print("Dimensions :", offre_demande.shape)
print("Communes sans MG :", (offre_demande["nb_medecins_generalistes"] == 0).sum())
print("Communes avec MG :", (offre_demande["nb_medecins_generalistes"] > 0).sum())


Dimensions : (34858, 16)
Communes sans MG : 25451
Communes avec MG : 9407


In [31]:
table_indicateur = offre_demande[[
    "CODGEO",
    "LIBGEO",
    "population",
    "demande_standardisee",
    "nb_medecins_generalistes"
]].copy()

table_indicateur.head()


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes
0,01001,L'Abergement-Clémenciat,859.0,867.710799,0.00
1,01002,L'Abergement-de-Varey,273.0,270.430838,0.00
2,01004,Ambérieu-en-Bugey,15554.0,14951.854488,16.85
3,01005,Ambérieux-en-Dombes,1917.0,1861.977364,1.00
4,01006,Ambléon,114.0,124.706348,0.00


In [32]:
table_indicateur.to_parquet(OUTPUT_LOCAL, index=False)
print("Fichier local enregistré :", OUTPUT_LOCAL)

if UPLOAD_S3:
    try:
        table_indicateur.to_parquet(OUTPUT_S3, index=False)
        print("Fichier envoyé sur MinIO :", OUTPUT_S3)
    except Exception as e:
        print("Upload MinIO impossible. Le fichier local reste disponible.")
        print("Erreur :", e)


Fichier local enregistré : data/processed/offre_demande_mg_commune.parquet
Fichier envoyé sur MinIO : s3://mangane/Projet_PDS_2026/processed/offre_demande_mg_commune.parquet


## Limites

- L'offre est un nombre de médecins généralistes et non un nombre annuel de consultations/visites.
- Un médecin exerçant dans plusieurs communes peut apparaître dans plusieurs communes.
- Les poids d'âge sont une approximation adaptée aux classes INSEE disponibles.
